# Colab BERT Runner

Run the project's BERT experiments on Google Colab and export the metrics, reports, predictions, and confusion matrix figures back to the local project.

## 1. Install Dependencies

Use a GPU runtime in Colab: `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
!pip install -q pandas scikit-learn matplotlib datasets transformers accelerate

## 2. Upload Project Files

Upload a zip that contains at least `src/`, and optionally `data/processed/` if you want to use the saved local CSV snapshots. If you do not upload `data/processed/`, the code will download MentalManip from Hugging Face.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()

for filename in uploaded:
    if filename.endswith(".zip"):
        with zipfile.ZipFile(filename, "r") as zip_ref:
            zip_ref.extractall(".")
        print(f"Extracted {filename}")

print("Top-level files:")
for path in sorted(Path(".").iterdir()):
    print(path)

## 3. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Run BERT Experiments

This notebook uses regular `bert-base-uncased` for the stage 1 task. If Colab runs out of memory, reduce both batch sizes to `4`.

In [ ]:
from pathlib import Path

from src.load_data import DEFAULT_CONFIG, load_dataset
from src.bert import (
    DEFAULT_LEARNING_RATE,
    DEFAULT_MAX_LENGTH,
    DEFAULT_WEIGHT_DECAY,
    print_summary,
    run_bert_classifier,
    save_classification_report,
    save_confusion_matrix_plot,
    save_metrics,
    save_predictions,
)

RUN_STAGE_1_BINARY = True
RUN_STAGE_2_TECHNIQUE = False

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 256
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 8
USE_CLASS_WEIGHTS_STAGE_1 = False
USE_CLASS_WEIGHTS_STAGE_2 = False
STAGE2_RARE_LABEL_MIN_COUNT = 2

Path("results/metrics").mkdir(parents=True, exist_ok=True)
Path("results/predictions").mkdir(parents=True, exist_ok=True)
Path("results/models").mkdir(parents=True, exist_ok=True)
Path("figures").mkdir(parents=True, exist_ok=True)


def run_stage(task: str):
    stage_name = "stage1_binary" if task == "binary" else "stage2_technique"
    rare_label_min_count = STAGE2_RARE_LABEL_MIN_COUNT if task == "technique" else None
    use_class_weights = USE_CLASS_WEIGHTS_STAGE_1 if task == "binary" else USE_CLASS_WEIGHTS_STAGE_2
    output_prefix = f"{stage_name}_weighted_bert" if use_class_weights else f"{stage_name}_bert"
    print(f"\n=== Running {stage_name} ===")
    print(f"Class-weighted loss: {use_class_weights}")

    df = load_dataset(
        task=task,
        config=DEFAULT_CONFIG,
        rare_label_min_count=None,
    )

    metrics, predictions, report_df = run_bert_classifier(
        df,
        stage_name=stage_name,
        model_name=MODEL_NAME,
        max_length=MAX_LENGTH,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=DEFAULT_LEARNING_RATE,
        weight_decay=DEFAULT_WEIGHT_DECAY,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        rare_label_min_count=rare_label_min_count,
        use_class_weights=use_class_weights,
        output_dir=Path("results/models") / output_prefix,
    )

    metrics_path = save_metrics(
        metrics,
        output_path=Path("results/metrics") / f"{output_prefix}_metrics.json",
    )
    predictions_path = save_predictions(
        predictions,
        output_path=Path("results/predictions") / f"{output_prefix}_predictions.csv",
    )
    report_path = save_classification_report(
        report_df,
        output_path=Path("results/metrics") / f"{output_prefix}_report.csv",
    )

    print_summary(metrics)
    print(f"Saved metrics to: {metrics_path}")
    print(f"Saved predictions to: {predictions_path}")
    print(f"Saved class report to: {report_path}")

    if metrics["confusion_matrix"]["manageable_for_display"]:
        confusion_path = save_confusion_matrix_plot(
            metrics["confusion_matrix"],
            output_path=Path("figures") / f"{output_prefix}_confusion_matrix.png",
            title=f"{stage_name} BERT Confusion Matrix",
        )
        print(f"Saved confusion matrix plot to: {confusion_path}")
    else:
        print("Skipped confusion matrix plot because there are too many labels.")

    return metrics


all_metrics = {}
if RUN_STAGE_1_BINARY:
    all_metrics["stage1_binary"] = run_stage("binary")
if RUN_STAGE_2_TECHNIQUE:
    all_metrics["stage2_technique"] = run_stage("technique")

## 5. Package Outputs

Download this zip and put the files back into the matching local project folders.

In [ ]:
!zip -r bert_colab_outputs.zip results figures -x "*/.ipynb_checkpoints/*"
files.download("bert_colab_outputs.zip")